# 第 7 章 练习题答案

> 精选 2 道核心练习，巩固指令微调理解。

## 练习 7.1：loss masking 验证

**题目**：指令微调时，为什么 prompt 部分要 mask（置 -100）？验证 mask 后 prompt 部分不产生梯度。

In [ ]:
import torch
import torch.nn.functional as F
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

def format_prompt(instr, inp):
    return f"### Instruction:\n{instr}\n### Input:\n{inp}\n### Response:\n"

tok = tiktoken.get_encoding("gpt2")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 64, "n_layers": 1, "n_heads": 4, "context_length": 32})
torch.manual_seed(0)
model = GPTModel(cfg)

# 构造一条指令数据
prompt = format_prompt("翻译", "你好")
prompt_ids = tok.encode(prompt)
resp_ids = tok.encode(" Hello")
input_ids = (prompt_ids + resp_ids)[:32]
targets = (input_ids[1:] + [tok.eot_token])[:32]
n_prompt = len(prompt_ids)
for i in range(min(n_prompt, len(targets))):
    targets[i] = -100   # mask prompt 部分
input_ids = input_ids + [50256]*(32-len(input_ids))
targets = targets + [-100]*(32-len(targets))

n_masked = (torch.tensor(targets) == -100).sum().item()
n_calc = 32 - n_masked
print(f"序列长度 32: mask {n_masked} 位置(prompt), 算 loss {n_calc} 位置(response)")
print("\n💡 mask 后 cross_entropy(ignore_index=-100) 自动跳过 prompt 部分，")
print("   模型只学'生成 Response'，不浪费容量学'复述指令'。")

## 练习 7.2：为什么需要固定模板？

**题目**：Alpaca 格式用 `### Instruction / ### Input / ### Response` 分隔符。如果不用固定模板会怎样？

**答案（概念题）**：
- 分隔符让模型学会在 `### Response:` 后**开始生成回答**，在前面的部分**停止**
- 没有固定模板，模型分不清「指令结束、响应开始」的边界，生成会混乱（可能续写指令而非回答）
- 固定模板也是推理时提取响应的依据：用 `### Response:\n` 分割就能拿到干净回答

In [ ]:
# 演示：用模板分隔符提取响应
def extract_response(full_text):
    """从生成结果里提取 Response 部分。"""
    return full_text.split("### Response:\n")[-1].strip()

# 模拟生成结果
generated = "### Instruction:\n翻译\n### Input:\n你好\n### Response:\n Hello"
response = extract_response(generated)
print(f"完整生成: {generated!r}")
print(f"提取响应: {response!r}  ← 干净的回答")
print("\n💡 固定模板让推理后处理极简：split 一下就拿到答案。")